# 🤖 01 — Fleet Discovery

Concurrent mDNS ping sweep across all known bot families.  
Results are merged with `fleet.yaml` metadata, displayed as a table, and saved to `ping_log.csv`.

---

## Setup

In [ ]:
import sys
from pathlib import Path

# Add fleet-manager to path so we can import without installing
REPO_ROOT = Path().resolve().parent
FLEET_MGR = REPO_ROOT / 'fleet-manager'
if str(FLEET_MGR) not in sys.path:
    sys.path.insert(0, str(FLEET_MGR))

DOCS = REPO_ROOT / 'docs'
FLEET_YAML = DOCS / 'fleet.yaml'
LOG_CSV    = DOCS / 'ping_log.csv'

print(f'Repo root : {REPO_ROOT}')
print(f'fleet.yaml: {FLEET_YAML}  (exists={FLEET_YAML.exists()})')

In [ ]:
from fleet_manager import scan, Bot, Fleet, load_fleet_yaml
print('fleet_manager imported ✓')

---
## 1 · Known Fleet (from fleet.yaml)

In [ ]:
# Load static fleet registry
yaml_meta = load_fleet_yaml(FLEET_YAML)

print(f'Loaded {len(yaml_meta)} entries from fleet.yaml\n')
for hostname, meta in yaml_meta.items():
    shipped = '⊗ SHIPPED' if meta.get('shipped') else '● active'
    print(f'  {hostname:<22} {shipped:<12} {meta.get("notes", "")[:60]}')

---
## 2 · Live Ping Sweep

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
BOT_FAMILIES = ['rfbot', 'mybot', 'carbot', 'paulbot', 'simplebot', 'dogbot']
SUFFIX_MAX   = 9      # ping {family}0.local … {family}9.local
WORKERS      = 20     # concurrent ping threads
TIMEOUT      = 1.0    # seconds per ping

import time
total = len(BOT_FAMILIES) * (SUFFIX_MAX + 1)
print(f'Scanning {total} targets with {WORKERS} workers…')

t0 = time.perf_counter()
fleet = scan(
    BOT_FAMILIES,
    max_suffix=SUFFIX_MAX,
    workers=WORKERS,
    timeout=TIMEOUT,
    yaml_path=FLEET_YAML,
)
elapsed = time.perf_counter() - t0

print(f'\n{fleet.summary()}')
print(f'Scan completed in {elapsed:.2f}s')

---
## 3 · Results Table

In [ ]:
# ── Pretty-print all bots ─────────────────────────────────────────────────────
COL = ('Hostname', 'Status', 'IP Address', 'Latency', 'Platform', 'Notes')
W   = (22, 9, 16, 10, 12, 40)

def _row(*cells):
    return '  '.join(str(c).ljust(w) for c, w in zip(cells, W))

print(_row(*COL))
print('─' * (sum(W) + 2 * len(W)))

STATUS_ICON = {'online': '●', 'offline': '○', 'shipped': '⊗', 'unknown': '?'}

for bot in fleet.bots:
    icon    = STATUS_ICON.get(bot.status, '?')
    latency = f'{bot.latency_ms:.1f} ms' if bot.latency_ms >= 0 else '—'
    print(_row(
        bot.hostname,
        f'{icon} {bot.status.capitalize()}',
        bot.ip if bot.ip != 'N/A' else '—',
        latency,
        bot.platform or '—',
        (bot.notes or '')[:40],
    ))

---
## 4 · Online Bots Summary

In [ ]:
print(f'Online bots ({len(fleet.online)}):')
for bot in fleet.online:
    print(f'  ● {bot.hostname:<22}  {bot.ip:<16}  {bot.latency_ms:.1f} ms')

if not fleet.online:
    print('  (none found — make sure bots are powered on and on the same network)')

---
## 5 · Visualise Fleet Status

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    # Count by status
    from collections import Counter
    counts = Counter(b.status for b in fleet.bots)
    labels = [k.capitalize() for k in counts]
    sizes  = list(counts.values())
    colors = {'Online': '#4ade80', 'Offline': '#f87171', 'Shipped': '#facc15', 'Unknown': '#94a3b8'}
    c      = [colors.get(l, '#94a3b8') for l in labels]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor('#0f172a')

    # Pie chart
    ax = axes[0]
    ax.set_facecolor('#0f172a')
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=c, autopct='%1.0f%%',
        startangle=90, pctdistance=0.75,
        textprops={'color': 'white', 'fontsize': 11},
    )
    for at in autotexts:
        at.set_color('#0f172a')
        at.set_fontweight('bold')
    ax.set_title('Fleet Status', color='white', fontsize=13, pad=12)

    # Latency bar chart (online bots only)
    ax2 = axes[1]
    ax2.set_facecolor('#1e293b')
    online = fleet.online
    if online:
        names = [b.hostname.replace('.local', '') for b in online]
        lats  = [b.latency_ms for b in online]
        bar_colors = ['#4ade80' if l < 10 else '#facc15' if l < 50 else '#f87171' for l in lats]
        bars = ax2.bar(names, lats, color=bar_colors, edgecolor='#334155')
        ax2.set_ylabel('Latency (ms)', color='#94a3b8')
        ax2.tick_params(colors='#94a3b8')
        ax2.spines[['top','right','bottom','left']].set_color('#334155')
        for spine in ax2.spines.values():
            spine.set_color('#334155')
        ax2.set_title('Round-trip Latency — Online Bots', color='white', fontsize=13, pad=12)
        plt.setp(ax2.get_xticklabels(), rotation=30, ha='right', color='#94a3b8')
    else:
        ax2.text(0.5, 0.5, 'No online bots', ha='center', va='center',
                 color='#94a3b8', fontsize=14, transform=ax2.transAxes)
        ax2.set_title('Round-trip Latency', color='white', fontsize=13)

    plt.tight_layout()
    plt.show()

except ImportError:
    print('matplotlib not installed — run: pip install matplotlib')
    print('Skipping chart.')

---
## 6 · Save to CSV

In [ ]:
log_path = fleet.to_csv(LOG_CSV, append=True)
print(f'Results appended to: {log_path}')

---
## 7 · Load History (optional — requires pandas)

In [ ]:
try:
    import pandas as pd

    df = pd.read_csv(LOG_CSV, parse_dates=['Timestamp'])
    print(f'Log file has {len(df)} rows across {df["Timestamp"].nunique()} scans.\n')
    print('Most recent scan:')
    latest = df[df['Timestamp'] == df['Timestamp'].max()]
    display(latest[['BotName', 'Status', 'IPAddress', 'LatencyMs']])

except ImportError:
    print('pandas not installed — run: pip install pandas')